# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [ ]:
import torch
import pandas as pd
from datasets import Dataset, Audio
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"

# 8-Bit Quantization Configuration for 12GB VRAM constraints
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

In [ ]:
print(f"Loading {model_id} in 8-bit precision...")
processor = AutoProcessor.from_pretrained(model_id)
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare model for gradient training
model = prepare_model_for_kbit_training(model)

#### Configuring QLoRA for Efficient Fine-Tuning

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

#### Dataset Formatting for Multimodal SFT

In [ ]:
def format_multimodal_dataset(csv_path):
    df = pd.read_csv(csv_path)
    # Filter out any rows that failed step 1.3
    df = df.dropna(subset=['Target_JSON', 'Target_GLaDOS_Response', 'Audio_File'])
    dataset_dict = {
        "audio": df["Audio_File"].tolist(),
        "messages": []
    }
    for _, row in df.iterrows():
        # Combine JSON and Persona text into the final ground truth target
        target_output = f"{row['Target_JSON']}\n\n{row['Target_GLaDOS_Response']}"
        # Voxtral Chat Template formatting
        conversation = [
            {
                "role": "user",
                "content": [{"type": "audio"}] # Audio tensor will be mapped here
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": target_output}]
            }
        ]
        dataset_dict["messages"].append(conversation)
    hf_dataset = Dataset.from_dict(dataset_dict)
    # Cast the audio column to HuggingFace's native Audio feature for automatic decoding
    hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=processor.feature_extractor.sampling_rate))
    return hf_dataset

print("Formatting multimodal dataset...")
train_dataset = format_multimodal_dataset("./data/multimodal_ground_truth_train.csv")

#### Supervised Fine-Tuning with TRL's SFTTrainer

In [ ]:
def collate_fn(examples):
    """Custom collator to process audio arrays and text templates into model inputs."""
    texts = [processor.apply_chat_template(ex["messages"], tokenize=False) for ex in examples]
    audios = [ex["audio"]["array"] for ex in examples]

    batch = processor(
        text=texts,
        audio=audios,
        sampling_rate=processor.feature_extractor.sampling_rate,
        return_tensors="pt",
        padding=True
    )

    # Mask user input and padding tokens from the loss calculation
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

training_args = SFTConfig(
    output_dir="./models/voxtral-glados-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500, # Adjust based on dataset size
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True, # Use bf16=True if training on the T4 Cluster
    remove_unused_columns=False, # Crucial: prevents HF from dropping the audio column
    dataset_kwargs={"skip_prepare_dataset": True}
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=lora_config,
)

In [ ]:
print("Initiating QLoRA Multimodal Alignment...")
trainer.train()

# Save the final adapter weights
trainer.model.save_pretrained("./models/voxtral-glados-final-adapters")
processor.save_pretrained("./models/voxtral-glados-final-adapters")
print("Training complete. Adapters saved.")